# Feature Maps Visualization Report

**Purpose**: Visualize learned features from CNN layers

**Outputs**: Layer-wise feature activation maps

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import yaml

sys.path.insert(0, '.')

from src.models.factory import ModelFactory
from src.data.dataloader import create_test_dataloader

# Configuration
config_path = 'configs/experiments/baseline.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = ModelFactory.create(
    name=config['model']['name'],
    num_classes=config['model']['num_classes'],
    hidden_features=config['model']['hidden_features']
)

checkpoint_path = Path('artifacts/checkpoints/last.pt')
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Model loaded")
else:
    print("⚠️  No checkpoint found")

model.to(device)
model.eval()

print(f"✓ Model architecture:")
print(model)

In [ ]:
# Get test data
test_loader = create_test_dataloader(
    data_dir=config['data']['data_dir'],
    batch_size=1
)

# Get first sample
X_sample, y_sample = next(iter(test_loader))
X_sample = X_sample.to(device)

print(f"✓ Sample image shape: {X_sample.shape}")
print(f"  Label: {y_sample.item()}")

# Forward pass and capture intermediate outputs
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach().cpu().numpy()
    return hook

# Register hooks for conv layers
hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        hook = module.register_forward_hook(get_activation(name))
        hooks.append(hook)

# Forward pass
with torch.no_grad():
    output = model(X_sample)

print(f"✓ Captured {len(activations)} layer activations")

# Remove hooks
for hook in hooks:
    hook.remove()

## 4. Input vs Output Comparison

In [ ]:
print("\n" + "="*70)
print("📊 Feature Maps Analysis Summary")
print("="*70)

print(f"\n🔍 Captured Layers:")
for layer_name, activation in activations.items():
    print(f"  {layer_name}: {activation.shape}")

print(f"\n💡 Insights:")
print(f"  • Early layers detect low-level features (edges, corners)")
print(f"  • Middle layers combine basic features")
print(f"  • Later layers extract high-level semantic features")
print(f"  • Visualization shows learned representations")

print("\n✅ Feature Maps Report Complete!")